<a href="https://colab.research.google.com/github/Saadd-x/FYP-Work/blob/main/Drone_YOLOv5n_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FYP Drone Detection — YOLOv5n Training

Clean notebook for training YOLOv5n on the Seraphim drone dataset.

This notebook will:
1. Mount Google Drive
2. Check GPU
3. Download the Seraphim dataset
4. Prepare the same one-class YOLO dataset with a reproducible 90/10 split
5. Save the prepared dataset as `drone_dataset.zip` to Drive BEFORE training
6. Clone YOLOv5
7. Train YOLOv5n for 30 epochs
8. Save `best.pt` and `last.pt` to Drive
9. Provide resume and video-testing cells

Your previous YOLOv5s run remains separate in `FYP_Drone/runs/test_drone/`.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
FYP_DIR = Path('/content/drive/MyDrive/FYP_Drone')
FYP_DIR.mkdir(parents=True, exist_ok=True)
print("FYP folder:", FYP_DIR)

Mounted at /content/drive
FYP folder: /content/drive/MyDrive/FYP_Drone


## 2. Check GPU

Before running this cell: **Runtime → Change runtime type → GPU**.

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Go to Runtime → Change runtime type → GPU.")

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Download the Seraphim dataset

In [3]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="lgrzybowski/seraphim-drone-detection-dataset",
    repo_type="dataset",
    local_dir="/content/seraphim"
)

print("Dataset downloaded.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Dataset downloaded.


## 4. Extract ZIP files if the dataset contains archives

In [4]:
import zipfile
from pathlib import Path

SOURCE = Path("/content/seraphim")
zip_files = list(SOURCE.rglob("*.zip"))

print("ZIP files found:", len(zip_files))

for z in zip_files:
    target = z.parent / z.stem
    if not target.exists():
        print("Extracting:", z.name)
        with zipfile.ZipFile(z, "r") as f:
            f.extractall(target)

print("Extraction complete.")

ZIP files found: 7
Extracting: batch_001.zip
Extracting: batch_001.zip
Extracting: batch_003.zip
Extracting: batch_004.zip
Extracting: batch_002.zip
Extracting: batch_001.zip
Extracting: batch_001.zip
Extraction complete.


## 5. Prepare the YOLO dataset

This uses one class (`drone`) and a fixed 90/10 train/validation split. Files are sorted before shuffling so the split is reproducible.

In [5]:
import random
import shutil
from pathlib import Path

SOURCE = Path("/content/seraphim")
OUT = Path("/content/drone_dataset")

source_images = SOURCE / "train" / "images"
source_labels = SOURCE / "train" / "labels"

# Fallback: locate a matching images/labels pair if the layout differs.
if not source_images.exists() or not source_labels.exists():
    image_dirs = [p for p in SOURCE.rglob("images") if p.is_dir()]
    label_dirs = [p for p in SOURCE.rglob("labels") if p.is_dir()]
    pairs_found = [(i, l) for i in image_dirs for l in label_dirs if i.parent == l.parent]

    if not pairs_found:
        raise FileNotFoundError(
            "Could not find a matching images/labels directory in /content/seraphim."
        )

    source_images, source_labels = pairs_found[0]

print("Images:", source_images)
print("Labels:", source_labels)

if OUT.exists():
    shutil.rmtree(OUT)

for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = sorted(p for p in source_images.iterdir() if p.suffix.lower() in image_exts)

pairs = []
for img in images:
    label = source_labels / f"{img.stem}.txt"
    if label.exists():
        pairs.append((img, label))

print("Images found:", len(images))
print("Matching image/label pairs:", len(pairs))

if not pairs:
    raise RuntimeError("No matching image/label pairs were found.")

random.seed(42)
random.shuffle(pairs)

cut = int(len(pairs) * 0.90)
train_pairs = pairs[:cut]
val_pairs = pairs[cut:]

def copy_pairs(items, split):
    for img, label in items:
        shutil.copy2(img, OUT / "images" / split / img.name)
        shutil.copy2(label, OUT / "labels" / split / label.name)

copy_pairs(train_pairs, "train")
copy_pairs(val_pairs, "val")

print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Dataset prepared.")

Images: /content/seraphim/train/images
Labels: /content/seraphim/train/labels
Images found: 0
Matching image/label pairs: 0


RuntimeError: No matching image/label pairs were found.

## 6. Create `data.yaml`

In [ ]:
data_yaml = OUT / "data.yaml"

data_yaml.write_text(
    "path: /content/drone_dataset\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 1\n"
    "names:\n"
    "  0: drone\n",
    encoding="utf-8"
)

print(data_yaml.read_text())

## 7. Verify the dataset

In [ ]:
train_count = len(list((OUT / "images" / "train").iterdir()))
val_count = len(list((OUT / "images" / "val").iterdir()))

print("Train images:", train_count)
print("Validation images:", val_count)
print("data.yaml:", data_yaml.exists())

assert train_count > 0 and val_count > 0 and data_yaml.exists()
print("Dataset verification passed.")

## 8. IMPORTANT — Save the prepared dataset to Drive

Do this BEFORE the long training. After this, future Colab runtimes can restore the dataset from Drive without downloading Seraphim again.

In [ ]:
import shutil
from pathlib import Path

zip_base = FYP_DIR / "drone_dataset"
zip_file = Path(str(zip_base) + ".zip")

if zip_file.exists():
    zip_file.unlink()

print("Creating ZIP. This may take a while...")
created = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir="/content",
    base_dir="drone_dataset"
)

size_gb = Path(created).stat().st_size / (1024**3)
print("Saved:", created)
print("ZIP size (GB):", round(size_gb, 2))

## 9. Clone YOLOv5 and install requirements

In [ ]:
%cd /content

!rm -rf /content/yolov5
!git clone https://github.com/ultralytics/yolov5.git

%cd /content/yolov5
!pip install -q -r requirements.txt

print("YOLOv5 ready.")

## 10. Train YOLOv5n for 30 epochs

This is a NEW experiment. It does not use the YOLOv5s checkpoint.

In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 30 \
    --data /content/drone_dataset/data.yaml \
    --weights yolov5n.pt \
    --name drone_v5n \
    --project /content/drive/MyDrive/FYP_Drone/runs \
    --save-period 1

## 11. Check the saved YOLOv5n files

In [ ]:
from pathlib import Path

RUN_DIR = Path("/content/drive/MyDrive/FYP_Drone/runs/drone_v5n")

print("Run folder:", RUN_DIR)
print("Exists:", RUN_DIR.exists())

if RUN_DIR.exists():
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file():
            print(p)

## 12. If Colab disconnects before training finishes

Do NOT restart from zero. Restore the saved dataset ZIP and resume from `last.pt`.

In [ ]:
# Restore the prepared dataset after a runtime reset:
!rm -rf /content/drone_dataset
!unzip -q "/content/drive/MyDrive/FYP_Drone/drone_dataset.zip" -d /content/

# After cloning/installing YOLOv5 again:
%cd /content/yolov5
!python train.py \
    --resume /content/drive/MyDrive/FYP_Drone/runs/drone_v5n/weights/last.pt

## 13. Test the best YOLOv5n model on a drone video

In [ ]:
%cd /content/yolov5

!python detect.py \
    --weights /content/drive/MyDrive/FYP_Drone/runs/drone_v5n/weights/best.pt \
    --source "/content/your_video.mp4" \
    --img 640 \
    --conf 0.25

## 14. Final FYP comparison

Keep the existing YOLOv5s experiment as the baseline:

`FYP_Drone/runs/test_drone/`

New YOLOv5n experiment:

`FYP_Drone/runs/drone_v5n/`

Compare:
- Precision
- Recall
- mAP@0.5
- mAP@0.5:0.95
- Model size
- FPS/latency
- FPGA resource utilization

Do not delete the YOLOv5s baseline.